<a href="https://colab.research.google.com/github/amoyag/Bioquimica_Ing_Proteinas/blob/main/ejercicio_design.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This exercise

In [1]:
# ============================================================================
# Setup
# ============================================================================

!pip install pyrosettacolabsetup
import pyrosettacolabsetup; pyrosettacolabsetup.install_pyrosetta()
import pyrosetta; pyrosetta.init()

from pyrosetta import *
from pyrosetta.rosetta import *
from pyrosetta.teaching import *
import pyrosetta.toolbox

# Init PyRosetta
pyrosetta.init("-ex1 -ex2aro")

!pip install py3Dmol
import py3Dmol
import tempfile
import urllib.request
import os

Mounted at /content/google_drive

Note that USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRE PURCHASE OF A LICENSE.
See https://github.com/RosettaCommons/rosetta/blob/main/LICENSE.md or email license@uw.edu for details.

Looking for compatible PyRosetta wheel file at google-drive/PyRosetta/colab.bin//wheels...
Found compatible wheel: /content/google_drive/MyDrive/PyRosetta/colab.bin/wheels//content/google_drive/MyDrive/PyRosetta/colab.bin/wheels/pyrosetta-2025.37+release.df75a9c48e-cp312-cp312-linux_x86_64.whl


┌───────────────────────────────────────────────────────────────────────────────┐
│                                  PyRosetta-4                                  │
│               Created in JHU by Sergey Lyskov and PyRosetta Team              │
│               (C) Copyright Rosetta Commons Member Institutions               │
│                                                                               │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRES PURCHASE OF A 

In [10]:
# Visualization

def display_protein(pose, style='stick'):
    """Visualiza una pose usando py3Dmol"""
    with tempfile.NamedTemporaryFile(suffix='.pdb') as tmp:
        pose.dump_pdb(tmp.name)
        with open(tmp.name, 'r') as pdb_file:
            pdb_string = pdb_file.read()

    viewer = py3Dmol.view(width=800, height=600)
    viewer.addModel(pdb_string, 'pdb')

    if style == 'stick':
        viewer.setStyle({'stick': {'colorscheme': 'greenCarbon'}})
    elif style == 'cartoon':
        viewer.setStyle({'cartoon': {'color': 'spectrum'}})
    else:
        viewer.setStyle({style: {'color': 'spectrum'}})

    viewer.zoomTo()
    viewer.show()

def compare_structures(pose1, pose2, names=['Structure 1', 'Structure 2']):
    """Compara dos estructuras lado a lado"""
    poses = [pose1, pose2]
    pdb_strings = []

    for pose in poses:
        with tempfile.NamedTemporaryFile(suffix='.pdb') as tmp:
            pose.dump_pdb(tmp.name)
            with open(tmp.name, 'r') as pdb_file:
                pdb_strings.append(pdb_file.read())

    viewer = py3Dmol.view(width=800, height=400, viewergrid=(1,2))

    for i, (pdb_string, name) in enumerate(zip(pdb_strings, names)):
        viewer.addModel(pdb_string, 'pdb', viewer=(0,i))
        viewer.setStyle({'cartoon': {'color': 'spectrum'}}, viewer=(0,i))
        viewer.addLabel(name, {'position':{'x':0,'y':0,'z':0},
                              'backgroundColor':'white', 'fontColor':'black'}, viewer=(0,i))

    viewer.zoomTo()
    viewer.show()

In [3]:
# ============================================================================
# DESCARGA Y PREPARACIÓN DE LA ESTRUCTURA 1GB1
# ============================================================================

def download_pdb_structure(pdb_id):
    """Descarga estructura del PDB"""
    url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
    filename = f"{pdb_id}.pdb"

    try:
        urllib.request.urlretrieve(url, filename)
        print(f"Estructura {pdb_id} descargada correctamente: {filename}")
        return filename
    except Exception as e:
        print(f"Error descargando {pdb_id}: {e}")
        return None

def extract_residue_range(pose, start_res, end_res):
    """Extrae un rango de residuos de una pose"""
    # Crear una nueva pose con la secuencia extraída
    sequence = ""
    for i in range(start_res, end_res + 1):
        if i <= pose.total_residue():
            sequence += pose.residue(i).name1()

    # Crear nueva pose desde secuencia
    new_pose = pose_from_sequence(sequence)

    # Copiar coordenadas del rango especificado
    for i, res_num in enumerate(range(start_res, end_res + 1)):
        if res_num <= pose.total_residue():
            for j in range(1, new_pose.residue(i+1).natoms() + 1):
                atom_name = new_pose.residue(i+1).atom_name(j)
                if pose.residue(res_num).has(atom_name):
                    old_xyz = pose.residue(res_num).xyz(atom_name)
                    new_pose.residue(i+1).set_xyz(atom_name, old_xyz)

    return new_pose

In [4]:
# ============================================================================
# PREPARACIÓN DEL β-HAIRPIN NATIVO
# ============================================================================

# Descargar estructura 1GB1
pdb_file = download_pdb_structure("1GB1")

if pdb_file and os.path.exists(pdb_file):
    # Cargar estructura completa
    full_pose = pose_from_pdb(pdb_file)
    print(f"Estructura completa cargada: {full_pose.total_residue()} residuos")
    print(f"Secuencia: {full_pose.sequence()}")

    # Extraer el β-hairpin (residuos 41-56)
    hairpin_native = extract_residue_range(full_pose, 41, 56)
    print(f"\nβ-hairpin extraído: {hairpin_native.total_residue()} residuos")
    print(f"Secuencia del β-hairpin: {hairpin_native.sequence()}")

    # Verificar que tenemos la secuencia correcta
    expected_sequence = "GEWTYDDATKTFTVTE"
    if hairpin_native.sequence() == expected_sequence:
        print("✓ Secuencia del β-hairpin correcta!")
    else:
        print(f"⚠ Secuencia inesperada. Esperada: {expected_sequence}")
        print(f"   Obtenida: {hairpin_native.sequence()}")

    # Guardar estructura nativa
    hairpin_native.dump_pdb("hairpin_native.pdb")
    print("\nEstructura nativa guardada como: hairpin_native.pdb")

else:
    print("Error: No se pudo descargar la estructura 1GB1")
    # Plan B: crear desde secuencia y usar estructura idealizada
    print("Creando estructura desde secuencia...")
    hairpin_native = pose_from_sequence("GEWTYDDATKTFTVTE")

Estructura 1GB1 descargada correctamente: 1GB1.pdb
core.chemical.GlobalResidueTypeSet: Finished initializing fa_standard residue type set.  Created 985 residue types
core.chemical.GlobalResidueTypeSet: Total time to initialize 1.31631 seconds.
core.import_pose.import_pose: File '1GB1.pdb' automatically determined to be of type PDB from contents.
core.io.pose_from_sfr.PoseFromSFRBuilder: [ WARNING ] discarding 3 atoms at position 1 in file 1GB1.pdb. Best match rsd_type:  MET:NtermProteinFull
core.io.pose_from_sfr.PoseFromSFRBuilder: [ WARNING ] discarding 3 atoms at position 57 in file 1GB1.pdb. Best match rsd_type:  MET:NtermProteinFull
core.io.pose_from_sfr.PoseFromSFRBuilder: [ WARNING ] discarding 3 atoms at position 113 in file 1GB1.pdb. Best match rsd_type:  MET:NtermProteinFull
core.io.pose_from_sfr.PoseFromSFRBuilder: [ WARNING ] discarding 3 atoms at position 169 in file 1GB1.pdb. Best match rsd_type:  MET:NtermProteinFull
core.io.pose_from_sfr.PoseFromSFRBuilder: [ WARNING ] d

In [5]:
# ============================================================================
# VISUALIZACIÓN DE LA ESTRUCTURA NATIVA
# ============================================================================

print("\n" + "="*50)
print("ESTRUCTURA NATIVA DEL β-HAIRPIN")
print("="*50)

display_protein(hairpin_native, style='cartoon')


ESTRUCTURA NATIVA DEL β-HAIRPIN


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [6]:
# ============================================================================
# ANÁLISIS ENERGÉTICO DE LA ESTRUCTURA NATIVA
# ============================================================================

# Configurar función de energía
scorefxn = get_fa_scorefxn()

native_energy = scorefxn(hairpin_native)
print(f"\nEnergía de la estructura nativa: {native_energy:.3f} REU")

# Mostrar desglose energético
print("\nDesglose energético de la estructura nativa:")
scorefxn.show(hairpin_native)

core.scoring.ScoreFunctionFactory: SCOREFUNCTION: ref2015
core.scoring.etable: Starting energy table calculation
core.scoring.etable: smooth_etable: changing atr/rep split to bottom of energy well
core.scoring.etable: smooth_etable: spline smoothing lj etables (maxdis = 6)
core.scoring.etable: smooth_etable: spline smoothing solvation etables (max_dis = 6)
core.scoring.etable: Finished calculating energy tables.
basic.io.database: Database file opened: scoring/score_functions/hbonds/ref2015_params/HBPoly1D.csv
basic.io.database: Database file opened: scoring/score_functions/hbonds/ref2015_params/HBFadeIntervals.csv
basic.io.database: Database file opened: scoring/score_functions/hbonds/ref2015_params/HBEval.csv
basic.io.database: Database file opened: scoring/score_functions/hbonds/ref2015_params/DonStrength.csv
basic.io.database: Database file opened: scoring/score_functions/hbonds/ref2015_params/AccStrength.csv
basic.io.database: Database file opened: scoring/score_functions/rama/fd/

In [7]:
# ============================================================================
# CREACIÓN DE VERSIONES PROBLEMÁTICAS
# ============================================================================

def create_problematic_variants(native_pose):
    """Crea diferentes versiones problemáticas del β-hairpin"""

    variants = {}

    # Variante 1: "Hydrophobic Disaster" - Romper core hidrofóbico
    var1 = native_pose.clone()
    # W43R, F52K - cambiar residuos hidrofóbicos por cargados
    pyrosetta.toolbox.mutants.mutate_residue(var1, 3, "R")  # W43R (posición 3 en el hairpin)
    pyrosetta.toolbox.mutants.mutate_residue(var1, 12, "K") # F52K (posición 12 en el hairpin)
    variants["hydrophobic_disaster"] = var1

    # Variante 2: "Charge Clash" - Crear repulsión electrostática
    var2 = native_pose.clone()
    # Y45E, T47E, K48R - acumular cargas negativas y positivas cercanas
    pyrosetta.toolbox.mutants.mutate_residue(var2, 5, "E")  # Y45E
    pyrosetta.toolbox.mutants.mutate_residue(var2, 7, "E")  # T47E
    pyrosetta.toolbox.mutants.mutate_residue(var2, 8, "R")  # K48R (mantener carga positiva)
    variants["charge_clash"] = var2

    # Variante 3: "Proline Breaker" - Introducir prolinas que rompen estructura β
    var3 = native_pose.clone()
    # Introducir prolinas en posiciones críticas de las hebras β
    pyrosetta.toolbox.mutants.mutate_residue(var3, 3, "P")  # W43P
    pyrosetta.toolbox.mutants.mutate_residue(var3, 12, "P") # F52P
    variants["proline_breaker"] = var3

    return variants

# Crear las variantes problemáticas
problematic_variants = create_problematic_variants(hairpin_native)

print("\n" + "="*50)
print("VARIANTES PROBLEMÁTICAS CREADAS")
print("="*50)

for name, variant in problematic_variants.items():
    energy = scorefxn(variant)
    print(f"\n{name.upper().replace('_', ' ')}:")
    print(f"Secuencia: {variant.sequence()}")
    print(f"Energía: {energy:.3f} REU (Δ = {energy - native_energy:.3f})")

    # Guardar estructura
    filename = f"hairpin_{name}.pdb"
    variant.dump_pdb(filename)
    print(f"Guardada como: {filename}")

core.scoring.ScoreFunctionFactory: SCOREFUNCTION: ref2015
core.pack.task: Packer task: initialize from command line()
core.pack.pack_rotamers: built 12 rotamers at 1 positions.
core.pack.interaction_graph.interaction_graph_factory: Instantiating PDInteractionGraph
core.scoring.ScoreFunctionFactory: SCOREFUNCTION: ref2015
core.pack.task: Packer task: initialize from command line()
core.pack.pack_rotamers: built 11 rotamers at 1 positions.
core.pack.interaction_graph.interaction_graph_factory: Instantiating PDInteractionGraph
core.scoring.ScoreFunctionFactory: SCOREFUNCTION: ref2015
core.pack.task: Packer task: initialize from command line()
core.pack.pack_rotamers: built 7 rotamers at 1 positions.
core.pack.interaction_graph.interaction_graph_factory: Instantiating PDInteractionGraph
core.scoring.ScoreFunctionFactory: SCOREFUNCTION: ref2015
core.pack.task: Packer task: initialize from command line()
core.pack.pack_rotamers: built 8 rotamers at 1 positions.
core.pack.interaction_graph.in

In [13]:
# ============================================================================
# RELAJACIÓN ESTRUCTURAL DE TODAS LAS VARIANTES
# ============================================================================

def setup_relax_protocol():
    """Configura protocolo FastRelax para relajar las estructuras"""
    # Configurar función de energía cartesiana
    scorefxn_relax = pyrosetta.create_score_function("ref2015_cart.wts")

    # Configurar MoveMap - permitir movimiento de backbone y sidechains
    mm = pyrosetta.rosetta.core.kinematics.MoveMap()
    mm.set_bb(True)   # Permitir movimiento del backbone
    mm.set_chi(True)  # Permitir movimiento de sidechains
    mm.set_jump(True) # Permitir movimiento de jumps (si los hay)

    # Configurar FastRelax
    relax = pyrosetta.rosetta.protocols.relax.FastRelax(scorefxn_in=scorefxn_relax, standard_repeats=1)
    relax.cartesian(True)  # Usar espacio cartesiano para mayor precisión
    relax.set_movemap(mm)
    relax.minimize_bond_angles(True)
    relax.minimize_bond_lengths(True)

    return relax, scorefxn_relax

def relax_structure(pose, name):
    """Relaja una estructura individual"""
    print(f"Relajando estructura: {name}")

    # Clonar para no modificar original
    relaxed_pose = pose.clone()

    # Configurar relax
    relax, scorefxn_relax = setup_relax_protocol()

    # Energía antes del relax
    energy_before = scorefxn(relaxed_pose)
    print(f"  Energía antes del relax: {energy_before:.3f} REU")

    # Aplicar relax
    relax.apply(relaxed_pose)

    # Energía después del relax
    energy_after = scorefxn(relaxed_pose)
    print(f"  Energía después del relax: {energy_after:.3f} REU")
    print(f"  Mejora energética: {energy_before - energy_after:.3f} REU")

    return relaxed_pose

In [14]:
# ============================================================================
# APLICAR RELAJACIÓN A TODAS LAS ESTRUCTURAS
# ============================================================================

print("\n" + "="*60)
print("RELAJACIÓN ESTRUCTURAL DE TODAS LAS VARIANTES")
print("="*60)

# Diccionario para almacenar estructuras relajadas
relaxed_structures = {}

# Relajar estructura nativa
print("\n🧬 ESTRUCTURA NATIVA")
print("-" * 30)
relaxed_structures["native"] = relax_structure(hairpin_native, "Nativa")

# Relajar todas las variantes problemáticas
for variant_name, variant_pose in problematic_variants.items():
    print(f"\n🧬 {variant_name.upper().replace('_', ' ')}")
    print("-" * 30)
    relaxed_structures[variant_name] = relax_structure(variant_pose, variant_name)

# Guardar estructuras relajadas
print("\n📁 Guardando estructuras relajadas...")
for name, pose in relaxed_structures.items():
    filename = f"relaxed_hairpin_{name}.pdb"
    pose.dump_pdb(filename)
    print(f"  {filename}")


RELAJACIÓN ESTRUCTURAL DE TODAS LAS VARIANTES

🧬 ESTRUCTURA NATIVA
------------------------------
Relajando estructura: Nativa
core.energy_methods.CartesianBondedEnergy: Initializing IdealParametersDatabase with default Ks=300 , 80 , 80 , 10 , 80
basic.io.database: Database file opened: scoring/score_functions/bondlength_bondangle/default-lengths.txt
core.energy_methods.CartesianBondedEnergy: Read 759 bb-independent lengths.
basic.io.database: Database file opened: scoring/score_functions/bondlength_bondangle/default-angles.txt
core.energy_methods.CartesianBondedEnergy: Read 1434 bb-independent angles.
basic.io.database: Database file opened: scoring/score_functions/bondlength_bondangle/default-torsions.txt
core.energy_methods.CartesianBondedEnergy: Read 1 bb-independent torsions.
basic.io.database: Database file opened: scoring/score_functions/bondlength_bondangle/default-improper.txt
core.energy_methods.CartesianBondedEnergy: Read 529 bb-independent improper tors.
protocols.relax.Re

In [15]:
# ============================================================================
# COMPARACIÓN ENERGÉTICA POST-RELAJACIÓN
# ============================================================================

def detailed_energy_analysis(pose, name):
    """Análisis energético detallado de una estructura"""
    total_energy = scorefxn(pose)

    # Crear diccionario con componentes energéticos principales
    energy_components = {}

    # Obtener energías por término usando el método show pero capturando output
    import io
    import sys
    from contextlib import redirect_stdout

    # Capturar output de scorefxn.show()
    f = io.StringIO()
    with redirect_stdout(f):
        scorefxn.show(pose)
    output = f.getvalue()

    # Parsear las líneas de energía más importantes
    for line in output.split('\n'):
        if 'fa_atr' in line:
            energy_components['fa_atr'] = float(line.split()[-1])
        elif 'fa_rep' in line:
            energy_components['fa_rep'] = float(line.split()[-1])
        elif 'fa_sol' in line:
            energy_components['fa_sol'] = float(line.split()[-1])
        elif 'fa_elec' in line:
            energy_components['fa_elec'] = float(line.split()[-1])
        elif 'hbond_sr_bb' in line:
            energy_components['hbond_sr_bb'] = float(line.split()[-1])
        elif 'hbond_lr_bb' in line:
            energy_components['hbond_lr_bb'] = float(line.split()[-1])
        elif 'hbond_sc' in line:
            energy_components['hbond_sc'] = float(line.split()[-1])

    return total_energy, energy_components

print("\n" + "="*70)
print("ANÁLISIS ENERGÉTICO COMPARATIVO POST-RELAJACIÓN")
print("="*70)

# Realizar análisis detallado de todas las estructuras
energy_results = {}

for name, pose in relaxed_structures.items():
    display_name = name.replace("_", " ").title() if name != "native" else "Native"
    total_energy, components = detailed_energy_analysis(pose, display_name)
    energy_results[name] = {'total': total_energy, 'components': components}



ANÁLISIS ENERGÉTICO COMPARATIVO POST-RELAJACIÓN
core.scoring.ScoreFunction: 
------------------------------------------------------------
 Scores                       Weight   Raw Score Wghtd.Score
------------------------------------------------------------
 fa_atr                       1.000     -56.871     -56.871
 fa_rep                       0.550       9.286       5.108
 fa_sol                       1.000      41.899      41.899
 fa_intra_rep                 0.005      37.751       0.189
 fa_intra_sol_xover4          1.000       4.794       4.794
 lk_ball_wtd                  1.000      -0.355      -0.355
 fa_elec                      1.000     -28.593     -28.593
 pro_close                    1.250       0.000       0.000
 hbond_sr_bb                  1.000      -2.771      -2.771
 hbond_lr_bb                  1.000      -5.823      -5.823
 hbond_bb_sc                  1.000      -2.439      -2.439
 hbond_sc                     1.000      -3.035      -3.035
 dslf_fa13         

In [16]:
# ============================================================================
# TABLA COMPARATIVA DE ENERGÍAS
# ============================================================================

print(f"\n{'='*70}")
print("TABLA COMPARATIVA DE ENERGÍAS (post-relajación)")
print(f"{'='*70}")

# Header de la tabla
print(f"{'Variante':<20} {'Energía Total':<15} {'Δ vs Nativa':<15} {'Secuencia':<20}")
print("-" * 70)

native_energy = energy_results["native"]["total"]

# Imprimir resultados
for name in ["native"] + list(problematic_variants.keys()):
    if name in energy_results:
        total_e = energy_results[name]["total"]
        delta_e = total_e - native_energy if name != "native" else 0.0

        if name == "native":
            sequence = hairpin_native.sequence()
            display_name = "Native"
        else:
            sequence = relaxed_structures[name].sequence()
            display_name = name.replace("_", " ").title()

        print(f"{display_name:<20} {total_e:<15.3f} {delta_e:<15.3f} {sequence:<20}")



TABLA COMPARATIVA DE ENERGÍAS (post-relajación)
Variante             Energía Total   Δ vs Nativa     Secuencia           
----------------------------------------------------------------------
Native               -22.272         0.000           GEWTYDDATKTFTVTE    
Hydrophobic Disaster -24.296         -2.024          GERTYDDATKTKTVTE    
Charge Clash         -18.735         3.537           GEWTEDERTKTFTVTE    
Proline Breaker      -28.013         -5.742          GEPTYDDATKTPTVTE    


In [17]:
# ============================================================================
# ANÁLISIS DETALLADO POR COMPONENTES ENERGÉTICOS
# ============================================================================

print(f"\n{'='*80}")
print("ANÁLISIS DETALLADO POR COMPONENTES ENERGÉTICOS")
print(f"{'='*80}")

# Lista de componentes energéticos importantes
important_terms = ['fa_atr', 'fa_rep', 'fa_sol', 'fa_elec', 'hbond_sr_bb', 'hbond_lr_bb', 'hbond_sc']

native_components = energy_results["native"]["components"]

print(f"{'Variante':<20}", end="")
for term in important_terms:
    print(f"{term:<12}", end="")
print()
print("-" * (20 + 12 * len(important_terms)))

# Datos para la nativa
print(f"{'Native':<20}", end="")
for term in important_terms:
    value = native_components.get(term, 0.0)
    print(f"{value:<12.3f}", end="")
print()

# Datos para cada variante problemática
for name in problematic_variants.keys():
    if name in energy_results:
        display_name = name.replace("_", " ").title()
        print(f"{display_name:<20}", end="")

        components = energy_results[name]["components"]
        for term in important_terms:
            value = components.get(term, 0.0)
            print(f"{value:<12.3f}", end="")
        print()

# ============================================================================
# IDENTIFICACIÓN DE PROBLEMAS PRINCIPALES
# ============================================================================

print(f"\n{'='*60}")
print("IDENTIFICACIÓN DE PROBLEMAS PRINCIPALES")
print(f"{'='*60}")

for name in problematic_variants.keys():
    if name in energy_results:
        display_name = name.replace("_", " ").upper()
        print(f"\n🔍 {display_name}:")
        print(f"   Secuencia: {relaxed_structures[name].sequence()}")

        total_penalty = energy_results[name]["total"] - native_energy
        print(f"   Penalización energética total: {total_penalty:.3f} REU")

        # Identificar los términos más problemáticos
        components = energy_results[name]["components"]
        problematic_terms = []

        for term in important_terms:
            if term in components and term in native_components:
                delta = components[term] - native_components[term]
                if abs(delta) > 5.0:  # Umbral para considerar problemático
                    problematic_terms.append((term, delta))

        # Ordenar por magnitud del problema
        problematic_terms.sort(key=lambda x: abs(x[1]), reverse=True)

        if problematic_terms:
            print("   Términos energéticos más problemáticos:")
            for term, delta in problematic_terms[:3]:  # Top 3
                direction = "↑" if delta > 0 else "↓"
                print(f"     • {term}: {delta:+.3f} REU {direction}")
        else:
            print("   No se detectaron términos energéticos muy problemáticos")



ANÁLISIS DETALLADO POR COMPONENTES ENERGÉTICOS
Variante            fa_atr      fa_rep      fa_sol      fa_elec     hbond_sr_bb hbond_lr_bb hbond_sc    
--------------------------------------------------------------------------------------------------------
Native              0.000       0.000       0.000       0.000       0.000       0.000       0.000       
Hydrophobic Disaster0.000       0.000       0.000       0.000       0.000       0.000       0.000       
Charge Clash        0.000       0.000       0.000       0.000       0.000       0.000       0.000       
Proline Breaker     0.000       0.000       0.000       0.000       0.000       0.000       0.000       

IDENTIFICACIÓN DE PROBLEMAS PRINCIPALES

🔍 HYDROPHOBIC DISASTER:
   Secuencia: GERTYDDATKTKTVTE
   Penalización energética total: -2.024 REU
   No se detectaron términos energéticos muy problemáticos

🔍 CHARGE CLASH:
   Secuencia: GEWTEDERTKTFTVTE
   Penalización energética total: 3.537 REU
   No se detectaron términos 

In [19]:
# ============================================================================
# VISUALIZACIÓN DE ESTRUCTURAS SUPERPUESTAS
# ============================================================================

def display_superimposed_structures(pose1, pose2, names=['Structure 1', 'Structure 2'],
                                   colors=['cyan', 'magenta'], style='cartoon'):
    """
    Visualiza dos estructuras superpuestas en el mismo visor

    Args:
        pose1, pose2: Poses de PyRosetta a comparar
        names: Lista con nombres para las estructuras
        colors: Lista con colores para cada estructura
        style: Estilo de visualización ('cartoon', 'stick', 'ribbon')
    """

    # Convertir poses a strings PDB
    pdb_strings = []
    for pose in [pose1, pose2]:
        with tempfile.NamedTemporaryFile(suffix='.pdb') as tmp:
            pose.dump_pdb(tmp.name)
            with open(tmp.name, 'r') as pdb_file:
                pdb_strings.append(pdb_file.read())

    # Crear visualizador
    viewer = py3Dmol.view(width=800, height=600)

    # Añadir ambas estructuras superpuestas
    for i, (pdb_string, name, color) in enumerate(zip(pdb_strings, names, colors)):
        viewer.addModel(pdb_string, 'pdb')

        # Configurar estilo según el tipo solicitado
        if style == 'cartoon':
            viewer.setStyle({'model': i}, {'cartoon': {'color': color, 'opacity': 0.8}})
        elif style == 'stick':
            viewer.setStyle({'model': i}, {'stick': {'colorscheme': color, 'radius': 0.3}})
        elif style == 'ribbon':
            viewer.setStyle({'model': i}, {'ribbon': {'color': color, 'opacity': 0.7}})

    # Añadir leyenda
    viewer.addLabel(f"{names[0]} ({colors[0]})",
                   {'position': {'x': -15, 'y': 10, 'z': 0},
                    'backgroundColor': colors[0], 'fontColor': 'white'})
    viewer.addLabel(f"{names[1]} ({colors[1]})",
                   {'position': {'x': -15, 'y': -5, 'z': 0},
                    'backgroundColor': colors[1], 'fontColor': 'white'})

    # Ajustar vista
    viewer.zoomTo()
    viewer.show()

def display_specific_variant_comparison(variant_name):
    """
    Función de conveniencia para comparar la nativa con una variante específica

    Args:
        variant_name: 'hydrophobic_disaster', 'charge_clash', o 'proline_breaker'
    """

    if variant_name not in relaxed_structures:
        print(f"Error: Variante '{variant_name}' no encontrada.")
        print(f"Variantes disponibles: {list(relaxed_structures.keys())}")
        return

    # Nombres más legibles
    variant_display_names = {
        'hydrophobic_disaster': 'Hydrophobic Disaster',
        'charge_clash': 'Charge Clash',
        'proline_breaker': 'Proline Breaker'
    }

    display_name = variant_display_names.get(variant_name, variant_name.title())

    # Información de las estructuras
    native_seq = relaxed_structures['native'].sequence()
    variant_seq = relaxed_structures[variant_name].sequence()
    native_energy = energy_results['native']['total']
    variant_energy = energy_results[variant_name]['total']
    delta_energy = variant_energy - native_energy

    print(f"\n" + "="*60)
    print(f"COMPARACIÓN: NATIVA vs {display_name.upper()}")
    print("="*60)
    print(f"Nativa:   {native_seq}  ({native_energy:.3f} REU)")
    print(f"Variante: {variant_seq}  ({variant_energy:.3f} REU)")
    print(f"Δ Energía: {delta_energy:+.3f} REU")

    # Identificar mutaciones
    mutations = []
    for i, (aa1, aa2) in enumerate(zip(native_seq, variant_seq)):
        if aa1 != aa2:
            mutations.append(f"{aa1}{i+1}{aa2}")

    if mutations:
        print(f"Mutaciones: {', '.join(mutations)}")
    print()

    # Visualizar superpuestas
    display_superimposed_structures(
        relaxed_structures['native'],
        relaxed_structures[variant_name],
        names=['Native', display_name],
        colors=['lightblue', 'salmon'],
        style='cartoon'
    )

# ============================================================================
# OPCIONES DE COMPARACIÓN INTERACTIVA
# ============================================================================

def show_comparison_menu():
    """Muestra menú de opciones para comparación"""
    print("\n" + "="*50)
    print("OPCIONES DE COMPARACIÓN ESTRUCTURAL")
    print("="*50)

    available_variants = [name for name in relaxed_structures.keys() if name != 'native']

    print("Variantes disponibles para comparar con la nativa:")
    for i, variant in enumerate(available_variants, 1):
        display_name = variant.replace('_', ' ').title()
        energy = energy_results[variant]['total']
        native_energy = energy_results['native']['total']
        delta = energy - native_energy
        print(f"{i}. {display_name:<20} (Δ = {delta:+.3f} REU)")

    print(f"\nPara visualizar una comparación, usa:")
    print(f"display_specific_variant_comparison('nombre_variante')")
    print(f"\nEjemplos:")
    for variant in available_variants:
        print(f"display_specific_variant_comparison('{variant}')")

# ============================================================================
# COMPARACIÓN RECOMENDADA: CHARGE CLASH
# ============================================================================

print("\n" + "="*60)
print("RECOMENDACIÓN: ANALIZAR CHARGE CLASH")
print("="*60)
print("La variante 'Charge Clash' es la única con penalización energética")
print("real después de la relajación (+3.537 REU), lo que la convierte")
print("en el mejor candidato para análisis de rescate.")
print()

# Mostrar automáticamente la comparación de Charge Clash
display_specific_variant_comparison('charge_clash')

# ============================================================================
# ANÁLISIS ESPECÍFICO DE CHARGE CLASH
# ============================================================================

def analyze_charge_clash_mutations():
    """Análisis detallado de las mutaciones en Charge Clash"""
    native_seq = "GEWTYDDATKTFTVTE"
    variant_seq = "GEWTEDERTKTFTVTE"

    print(f"\n" + "="*50)
    print("ANÁLISIS DETALLADO: CHARGE CLASH")
    print("="*50)

    print("Secuencia nativa: GEWTYDDATKTFTVTE")
    print("Secuencia variant: GEWTEDERTKTFTVTE")
    print("                   ....ED.R........")
    print()

    print("Mutaciones identificadas:")
    print("• Y45E (posición 5): Tirosina → Ácido Glutámico")
    print("• T47E (posición 7): Treonina → Ácido Glutámico")
    print("• K48R (posición 8): Lisina → Arginina")
    print()

    print("Problemas potenciales:")
    print("1. Acumulación de cargas negativas (E45, E47)")
    print("2. Repulsión electrostática entre residuos cercanos")
    print("3. Posible desestabilización del turn region")
    print("4. Pérdida de interacciones favorables (Y45 aromática)")
    print()

    print("Estrategias de rescate sugeridas:")
    print("• Neutralizar una de las cargas negativas (E→Q o E→N)")
    print("• Restaurar carácter hidrofóbico en posición 45 (E→F o E→Y)")
    print("• Optimizar el turn region manteniendo la estructura β-hairpin")

# Ejecutar análisis automáticamente
analyze_charge_clash_mutations()


RECOMENDACIÓN: ANALIZAR CHARGE CLASH
La variante 'Charge Clash' es la única con penalización energética
real después de la relajación (+3.537 REU), lo que la convierte
en el mejor candidato para análisis de rescate.


COMPARACIÓN: NATIVA vs CHARGE CLASH
Nativa:   GEWTYDDATKTFTVTE  (-22.272 REU)
Variante: GEWTEDERTKTFTVTE  (-18.735 REU)
Δ Energía: +3.537 REU
Mutaciones: Y5E, D7E, A8R



3Dmol.js failed to load for some reason. Please check your browser console for error messages.


ANÁLISIS DETALLADO: CHARGE CLASH
Secuencia nativa: GEWTYDDATKTFTVTE
Secuencia variant: GEWTEDERTKTFTVTE
                   ....ED.R........

Mutaciones identificadas:
• Y45E (posición 5): Tirosina → Ácido Glutámico
• T47E (posición 7): Treonina → Ácido Glutámico
• K48R (posición 8): Lisina → Arginina

Problemas potenciales:
1. Acumulación de cargas negativas (E45, E47)
2. Repulsión electrostática entre residuos cercanos
3. Posible desestabilización del turn region
4. Pérdida de interacciones favorables (Y45 aromática)

Estrategias de rescate sugeridas:
• Neutralizar una de las cargas negativas (E→Q o E→N)
• Restaurar carácter hidrofóbico en posición 45 (E→F o E→Y)
• Optimizar el turn region manteniendo la estructura β-hairpin
